# 🎯 Cache Strategies

**Advanced caching patterns for LLM applications**

---

## 📋 Overview

**What you'll learn:**
- Write-through vs write-back
- Cache-aside pattern
- Multi-level caching
- Partial response caching
- Cache preloading

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from typing import Optional, Dict, Any
import time
import hashlib
import json

print("✅ Setup complete")

## 🎯 Cache Patterns Overview

### Common Caching Patterns:

```
1. Cache-Aside (Lazy Loading)
   - Read: Check cache → Miss → Load from DB → Store in cache
   - Most common pattern
   
2. Write-Through
   - Write to cache AND database simultaneously
   - Ensures consistency
   
3. Write-Back (Write-Behind)
   - Write to cache → Async write to database
   - Better performance, eventual consistency
   
4. Read-Through
   - Cache handles loading from database
   - Transparent to application
   
5. Refresh-Ahead
   - Proactively refresh cache before expiration
   - Prevent cache misses
```

## 📖 Cache-Aside Pattern

In [ ]:
class CacheAsidePattern:
    """Cache-aside (lazy loading) pattern."""
    
    def __init__(self, cache, database):
        self.cache = cache
        self.database = database
    
    def get(self, key: str) -> Optional[Any]:
        """Get value with cache-aside pattern."""
        
        # 1. Try cache first
        value = self.cache.get(key)
        if value is not None:
            print(f"  ✅ Cache hit: {key}")
            return value
        
        print(f"  ❌ Cache miss: {key}")
        
        # 2. Load from database
        value = self.database.get(key)
        if value is None:
            return None
        
        # 3. Store in cache for next time
        self.cache.set(key, value, ttl=3600)
        print(f"  💾 Cached: {key}")
        
        return value
    
    def set(self, key: str, value: Any):
        """Write to database (cache-aside doesn't update cache on write)."""
        
        # Write to database
        self.database.set(key, value)
        
        # Invalidate cache (optional)
        self.cache.delete(key)
        print(f"  💾 Wrote to DB and invalidated cache: {key}")

print("""
📖 Cache-Aside Pattern

Flow:
  READ:
    1. Check cache
    2. If miss → Load from DB
    3. Store in cache
    4. Return value
  
  WRITE:
    1. Write to database
    2. Invalidate cache entry

✅ Pros:
  - Simple to implement
  - Cache only what's needed
  - Resilient (cache can fail)

❌ Cons:
  - Initial request is slow (cache miss)
  - Possible stale data
  - Extra code in application
""")

## 🔄 Write-Through Pattern

In [ ]:
class WriteThroughCache:
    """Write-through caching pattern."""
    
    def __init__(self, cache, database):
        self.cache = cache
        self.database = database
    
    def get(self, key: str) -> Optional[Any]:
        """Read from cache (should always be there)."""
        return self.cache.get(key)
    
    def set(self, key: str, value: Any):
        """Write to both cache and database."""
        
        # Write to database first
        self.database.set(key, value)
        print(f"  💾 Wrote to database: {key}")
        
        # Then write to cache
        self.cache.set(key, value)
        print(f"  💾 Wrote to cache: {key}")
        
        # Cache and DB are always in sync!

print("""
🔄 Write-Through Pattern

Flow:
  WRITE:
    1. Write to database
    2. Write to cache
    3. Return success
  
  READ:
    1. Read from cache
    2. Return value (always in cache)

✅ Pros:
  - Cache always consistent
  - Read performance excellent
  - Simple read logic

❌ Cons:
  - Slower writes (2 operations)
  - Cache can have unused data
  - If cache fails, writes fail
""")

## ⚡ Write-Back Pattern

In [ ]:
print("""
import asyncio
from collections import deque

class WriteBackCache:
    \"\"\"Write-back (write-behind) caching pattern.\"\"\" 
    
    def __init__(self, cache, database, flush_interval=5):
        self.cache = cache
        self.database = database
        self.flush_interval = flush_interval
        self.write_queue = deque()  # Queue of pending writes
        self.running = False
    
    def get(self, key: str) -> Optional[Any]:
        \"\"\"Read from cache.\"\"\" 
        return self.cache.get(key)
    
    def set(self, key: str, value: Any):
        \"\"\"Write to cache immediately, queue DB write.\"\"\" 
        
        # Write to cache immediately (fast!)
        self.cache.set(key, value)
        print(f"  ⚡ Wrote to cache: {key}")
        
        # Queue database write (async)
        self.write_queue.append((key, value))
        print(f"  📝 Queued for DB write: {key}")
    
    async def flush_worker(self):
        \"\"\"Background worker to flush writes to database.\"\"\" 
        
        while self.running:
            await asyncio.sleep(self.flush_interval)
            
            if self.write_queue:
                print(f"\n  🔄 Flushing {len(self.write_queue)} writes to database...")
                
                while self.write_queue:
                    key, value = self.write_queue.popleft()
                    self.database.set(key, value)
                    print(f"    💾 Flushed: {key}")
                
                print("  ✅ Flush complete\n")
    
    async def start(self):
        \"\"\"Start background flush worker.\"\"\" 
        self.running = True
        asyncio.create_task(self.flush_worker())
    
    def stop(self):
        \"\"\"Stop and flush remaining writes.\"\"\" 
        self.running = False
        
        # Flush any remaining writes
        while self.write_queue:
            key, value = self.write_queue.popleft()
            self.database.set(key, value)

⚡ Write-Back Pattern

Flow:
  WRITE:
    1. Write to cache (immediate)
    2. Queue DB write (async)
    3. Return success (fast!)
    4. Background worker flushes to DB
  
  READ:
    1. Read from cache
    2. Return value

✅ Pros:
  - Fastest write performance
  - Batch DB writes (efficient)
  - Reduced DB load

❌ Cons:
  - Risk of data loss (cache failure)
  - Eventual consistency
  - Complex implementation
  - Need backup/recovery strategy
""")

## 🏗️ Multi-Level Caching

In [ ]:
class MultiLevelCache:
    """Multi-level caching (L1 → L2 → Database)."""
    
    def __init__(self, l1_cache, l2_cache, database):
        """
        Args:
            l1_cache: Fast, small cache (in-memory)
            l2_cache: Slower, larger cache (Redis)
            database: Slowest, permanent storage
        """
        self.l1 = l1_cache  # In-memory (100MB, 1ms)
        self.l2 = l2_cache  # Redis (10GB, 5ms)
        self.db = database  # PostgreSQL (∞, 50ms)
        
        self.stats = {
            'l1_hits': 0,
            'l2_hits': 0,
            'db_hits': 0
        }
    
    def get(self, key: str) -> Optional[Any]:
        """Get value from multi-level cache."""
        
        # Level 1: In-memory (fastest)
        value = self.l1.get(key)
        if value is not None:
            self.stats['l1_hits'] += 1
            print(f"  ⚡ L1 hit (1ms): {key}")
            return value
        
        # Level 2: Redis (fast)
        value = self.l2.get(key)
        if value is not None:
            self.stats['l2_hits'] += 1
            print(f"  🔷 L2 hit (5ms): {key}")
            
            # Promote to L1
            self.l1.set(key, value, ttl=300)  # Short TTL in L1
            return value
        
        # Level 3: Database (slow)
        value = self.db.get(key)
        if value is not None:
            self.stats['db_hits'] += 1
            print(f"  💾 DB hit (50ms): {key}")
            
            # Store in both caches
            self.l2.set(key, value, ttl=3600)  # 1 hour in L2
            self.l1.set(key, value, ttl=300)   # 5 min in L1
            return value
        
        return None
    
    def set(self, key: str, value: Any):
        """Write to all levels."""
        
        # Write to database (authoritative)
        self.db.set(key, value)
        
        # Update caches
        self.l2.set(key, value, ttl=3600)
        self.l1.set(key, value, ttl=300)
    
    def get_stats(self) -> dict:
        """Get cache hit statistics."""
        total = sum(self.stats.values())
        
        if total == 0:
            return self.stats
        
        return {
            **self.stats,
            'l1_rate': f"{self.stats['l1_hits']/total*100:.1f}%",
            'l2_rate': f"{self.stats['l2_hits']/total*100:.1f}%",
            'db_rate': f"{self.stats['db_hits']/total*100:.1f}%"
        }

print("""
🏗️ Multi-Level Cache Strategy

Levels:
  L1: In-Memory  (100MB,  1ms latency)  → Hot data
  L2: Redis      (10GB,   5ms latency)  → Warm data
  L3: Database   (∞,     50ms latency)  → All data

Performance:
  - 70% L1 hits →  1ms avg
  - 20% L2 hits →  5ms avg
  - 10% DB hits → 50ms avg
  - Overall: ~6ms average latency

  vs. No caching: 50ms every request
  Improvement: 8x faster!
""")

## 🎯 Partial Response Caching

In [ ]:
print("""
# Cache parts of LLM responses

class PartialResponseCache:
    \"\"\"Cache reusable parts of prompts/responses.\"\"\" 
    
    def __init__(self, cache):
        self.cache = cache
    
    def get_completion(self, system_prompt: str, user_query: str):
        \"\"\"Get completion with partial caching.\"\"\" 
        
        # Cache key for system prompt processing
        system_key = f"system:{hash(system_prompt)}"
        
        # Check if we've seen this system prompt
        system_context = self.cache.get(system_key)
        
        if system_context:
            print("  ✅ System prompt cached (save tokens!)")
        else:
            print("  ❌ Processing system prompt...")
            # Process and cache system prompt
            system_context = process_system_prompt(system_prompt)
            self.cache.set(system_key, system_context, ttl=86400)
        
        # Check full query cache
        full_key = f"query:{hash(system_prompt + user_query)}"
        cached_response = self.cache.get(full_key)
        
        if cached_response:
            print("  ✅ Full response cached")
            return cached_response
        
        # Generate response using cached system context
        response = call_llm(system_context, user_query)
        
        # Cache full response
        self.cache.set(full_key, response, ttl=3600)
        
        return response

# Example: Multi-turn conversation caching

class ConversationCache:
    \"\"\"Cache conversation history efficiently.\"\"\" 
    
    def __init__(self, cache):
        self.cache = cache
    
    def get_response(
        self,
        conversation_id: str,
        messages: list
    ) -> str:
        \"\"\"Get response with conversation caching.\"\"\" 
        
        # Create cache key from conversation history
        history_key = f"conv:{conversation_id}:{hash(json.dumps(messages))}"
        
        # Check if exact conversation state is cached
        cached = self.cache.get(history_key)
        if cached:
            return cached
        
        # Check if earlier part of conversation is cached
        if len(messages) > 1:
            prev_messages = messages[:-1]
            prev_key = f"conv:{conversation_id}:{hash(json.dumps(prev_messages))}"
            prev_state = self.cache.get(prev_key)
            
            if prev_state:
                print("  ♻️ Reusing cached conversation state")
                # Only process new message
                response = call_llm_incremental(prev_state, messages[-1])
            else:
                # Process full conversation
                response = call_llm(messages)
        else:
            response = call_llm(messages)
        
        # Cache this state
        self.cache.set(history_key, response, ttl=1800)
        
        return response

💡 Partial Caching Strategies:

1. System Prompts
   - Cache processed system prompts
   - Reuse across many queries
   - Save tokens and latency

2. Conversation State
   - Cache conversation prefixes
   - Only process new messages
   - Incremental computation

3. Few-Shot Examples
   - Cache example embeddings
   - Reuse for similar tasks
   - Consistent formatting

4. Template Outputs
   - Cache common output patterns
   - Fill in variables
   - Fast personalization
""")

## ✅ Summary

### Cache Patterns Comparison:

| Pattern | Use Case | Consistency | Performance |
|---------|----------|-------------|-------------|
| **Cache-Aside** | General purpose | Eventual | Good |
| **Write-Through** | Read-heavy | Strong | Good reads, slow writes |
| **Write-Back** | Write-heavy | Eventual | Excellent |
| **Read-Through** | Simple apps | Strong | Good |
| **Multi-Level** | High traffic | Varies | Excellent |

### When to Use:

**Cache-Aside:**
```python
# Most common, simple, flexible
value = cache.get(key)
if not value:
    value = db.get(key)
    cache.set(key, value)

✅ Use for:
  - LLM response caching
  - General APIs
  - Flexible caching needs
```

**Write-Through:**
```python
# Always keep cache updated
db.set(key, value)
cache.set(key, value)

✅ Use for:
  - Strong consistency required
  - Read-heavy workloads
  - Critical data
```

**Write-Back:**
```python
# Fast writes, async DB updates
cache.set(key, value)
queue_db_write(key, value)

✅ Use for:
  - High write throughput
  - Analytics, metrics
  - Batch operations
```

**Multi-Level:**
```python
# L1 (memory) → L2 (Redis) → DB
value = l1.get() or l2.get() or db.get()

✅ Use for:
  - Hot data access
  - Large scale systems
  - Performance critical
```

### Best Practices:

**1. Choose Right Pattern**
```python
# Start simple (cache-aside)
# Optimize later if needed
```

**2. Set Appropriate TTLs**
```python
l1.set(key, value, ttl=300)    # 5 min (hot)
l2.set(key, value, ttl=3600)   # 1 hour (warm)
```

**3. Monitor Cache Performance**
```python
metrics = {
    'hit_rate': 0.85,  # 85% hits
    'avg_latency': 5,  # 5ms
    'size': 1000000    # 1M entries
}
```

**4. Handle Cache Failures**
```python
try:
    value = cache.get(key)
except CacheError:
    value = db.get(key)  # Fallback
```

### Next: `09_caching/04_distributed_caching.ipynb`